In [1]:
!pip install git+https://github.com/WildlifeDatasets/wildlife-datasets@develop
!pip install git+https://github.com/lightshing/wildlife-tools
!pip install pytorch-metric-learning
!pip install umap-learn
!pip install umap
!pip install record-keeper
!pip install faiss-cpu

  Cloning https://github.com/WildlifeDatasets/wildlife-datasets (to revision develop) to /tmp/pip-req-build-73xjo5ll
  Running command git clone --filter=blob:none --quiet https://github.com/WildlifeDatasets/wildlife-datasets /tmp/pip-req-build-73xjo5ll
  Running command git checkout -b develop --track origin/develop
  Switched to a new branch 'develop'
  Branch 'develop' set up to track remote branch 'develop' from 'origin'.
  Resolved https://github.com/WildlifeDatasets/wildlife-datasets to commit 753d9bf64861c3e17011136b3436bf58bf02317f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 6.7 MB/s eta 0:00:00
  Created wheel for wildlife-datasets: filename=wildlife_datasets-1.0.6-py3-none-any.whl size=88076 sha256=cf02a5e16c1e6d3bcee8a6055f0bd2e2b35ee9b03e9c0818f383265d0cfae851
  Stored in directory: /tmp/pip-ephem-wheel-cache-m54gti7_/wheels/a

In [2]:
import os
import numpy as np
import pandas as pd
import timm
import torchvision.transforms as T
import torch
from torch import nn
from torch import optim
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from wildlife_datasets.datasets import AnimalCLEF2025
from wildlife_datasets import splits
from sklearn.preprocessing import LabelEncoder

%matplotlib inline
import logging



from pytorch_metric_learning import distances, reducers

import pytorch_metric_learning
from pytorch_metric_learning import losses, miners, samplers
from pytorch_metric_learning.utils.accuracy_calculator import AccuracyCalculator
import pytorch_metric_learning.utils.logging_presets as LP
from pytorch_metric_learning.trainers import TrainWithClassifier
from pytorch_metric_learning.testers import BaseTester
import torch
import os


logging.getLogger().setLevel(logging.INFO)
logging.info("VERSION %s" % pytorch_metric_learning.__version__)

2025-05-03 18:25:26.685205: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746296726.875448      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746296726.928503      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
root = '/kaggle/input/offline-imgaug/aug'
out_root = '/kaggle/working'
checkpoint_out_path = '/kaggle/working/model_checkpoints'

In [4]:
set_split_seed = 333
training_seed = 666
batch_size = 16
Vmargin = 0.22

In [5]:
name = 'hf-hub:BVRA/MegaDescriptor-L-384'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mega = timm.create_model(name, num_classes=0, pretrained=True)
print(device)
mega = mega.to(device)

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

cuda


In [6]:
transform_mega_ori = timm.data.create_transform(
    **timm.data.resolve_data_config(mega.pretrained_cfg)
)
transform_mega_ori

Compose(
    Resize(size=426, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(384, 384))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)

In [7]:
transform = T.Compose([
    T.ToTensor(),
    *transform_mega_ori.transforms
    ])

In [8]:
# Loading the dataset
dataset = AnimalCLEF2025(root, transform=transform, load_label=True)
dataset_database = dataset.get_subset(dataset.metadata['split'] == 'database')
dataset_query = dataset.get_subset(dataset.metadata['split'] == 'query')
n_query = len(dataset_query)

In [9]:
dataset.metadata[['dataset', 'split']].value_counts()

dataset           split   
SeaTurtleID2022   database    8821
SalamanderID2025  database    3064
LynxID2025        database    3006
                  query        946
SalamanderID2025  query        689
SeaTurtleID2022   query        500
Name: count, dtype: int64

In [10]:
dataset_database.df

,image_id,identity,path,date,orientation,species,split,dataset
0,0,LynxID2025_lynx_37,images/LynxID2025/database/0.jpg,NaN,right,lynx,database,LynxID2025
1,1,LynxID2025_lynx_37,images/LynxID2025/database/1.jpg,NaN,left,lynx,database,LynxID2025
2,2,LynxID2025_lynx_49,images/LynxID2025/database/2.jpg,NaN,left,lynx,database,LynxID2025
3,4,LynxID2025_lynx_13,images/LynxID2025/database/4.jpg,NaN,right,lynx,database,LynxID2025
4,6,LynxID2025_lynx_07,images/LynxID2025/database/6.jpg,NaN,left,lynx,database,LynxID2025
...,...,...,...,...,...,...,...,...
14886,17021,SalamanderID2025_647,images/SalamanderID2025/database/images/17021.jpg,2024-10-02,top,NaN,database,SalamanderID2025
14887,17022,SalamanderID2025_507,images/SalamanderID2025/database/images/17022.jpg,2023-09-22,right,NaN,database,SalamanderID2025
14888,17023,SalamanderID2025_507,images/SalamanderID2025/database/images/17023.jpg,2023-09-22,right,NaN,database,SalamanderID2025
14889,17024,SalamanderID2025_507,images/SalamanderID2025/database/images/17024.jpg,2023-09-22,right,NaN,database,SalamanderID2025


### 分为4个set，open open close
train/test/thresholdSet/testSet

In [11]:
df_database = dataset_database.df
df_database.dataset.value_counts()

dataset
SeaTurtleID2022     8821
SalamanderID2025    3064
LynxID2025          3006
Name: count, dtype: int64

In [12]:
openSplitter = splits.OpenSetSplit(0.9, 0.1, seed=set_split_seed)
for idx_train, idx_test in openSplitter.split(df_database):
    splits.analyze_split(df_database, idx_train, idx_test)

Split: time-unaware open-set
Samples: train/test/unassigned/total = 12399/2492/0/14891
Classes: train/test/unassigned/total = 1002/1102/0/1102
Samples: train only/test only        = 0/1490
Classes: train only/test only/joint  = 0/100/1002

Fraction of train set     = 83.27%
Fraction of test set only = 10.01%


In [13]:
df_train, df_test = df_database.loc[idx_train], df_database.loc[idx_test]
print(len(df_train))
print(len(df_test))

12399
2492


In [14]:
availableSet = dataset_database.get_subset(df_train.index)
testSet = dataset_database.get_subset(df_test.index)

In [15]:
df_availableSet = availableSet.df
df_availableSet.dataset.value_counts()

dataset
SeaTurtleID2022     7617
LynxID2025          2533
SalamanderID2025    2249
Name: count, dtype: int64

In [16]:
openSplitter = splits.OpenSetSplit(0.9, 0.1, seed=set_split_seed)
for idx_train, idx_test in openSplitter.split(df_availableSet):
    splits.analyze_split(df_availableSet, idx_train, idx_test)

Split: time-unaware open-set
Samples: train/test/unassigned/total = 10244/2155/0/12399
Classes: train/test/unassigned/total = 912/1002/0/1002
Samples: train only/test only        = 0/1243
Classes: train only/test only/joint  = 0/90/912

Fraction of train set     = 82.62%
Fraction of test set only = 10.03%


In [17]:
df_train, df_test = df_availableSet.loc[idx_train], df_availableSet.loc[idx_test]
print(len(df_train))
print(len(df_test))

10244
2155


In [18]:
trainSet = availableSet.get_subset(df_train.index)
thresholdSet = availableSet.get_subset(df_test.index)

#### 标签转换为数字

In [19]:
df_database_training = trainSet.df

df_database_training['name'] = df_database_training['identity'].copy()

encoder = LabelEncoder()
df_database_training['identity'] = encoder.fit_transform(df_database_training['identity'])

trainSet.df = df_database_training

In [20]:
df_database_training.to_csv("/kaggle/working/mm.csv", index=False, encoding='utf-8-sig')

In [21]:
df_database_training.dataset.value_counts()

dataset
SeaTurtleID2022     6590
LynxID2025          2101
SalamanderID2025    1553
Name: count, dtype: int64

In [22]:
closeSplitter = splits.ClosedSetSplit(0.9, seed=set_split_seed)
for idx_train, idx_test in closeSplitter.split(df_database_training):
    splits.analyze_split(df_database_training, idx_train, idx_test)

Split: time-unaware closed-set
Samples: train/test/unassigned/total = 8802/1442/0/10244
Classes: train/test/unassigned/total = 912/912/0/912
Samples: train only/test only        = 0/0
Classes: train only/test only/joint  = 0/0/912

Fraction of train set     = 85.92%
Fraction of test set only = 0.00%


In [23]:
df_training, df_val = df_database_training.loc[idx_train], df_database_training.loc[idx_test]
print(len(df_training))
print(len(df_val))

8802
1442


In [24]:
training = trainSet.get_subset(df_training.index)
test = trainSet.get_subset(df_val.index)

In [25]:
training_labels = training.df["identity"].tolist()

In [26]:
df_of_training = training.df
df_of_training.dataset.value_counts()

dataset
SeaTurtleID2022     5859
LynxID2025          1874
SalamanderID2025    1069
Name: count, dtype: int64

In [27]:
closeSplitter = splits.ClosedSetSplit(0.1, seed=set_split_seed)
for idx_train, idx_test in closeSplitter.split(df_of_training):
    splits.analyze_split(df_of_training, idx_train, idx_test)

Split: time-unaware closed-set
Samples: train/test/unassigned/total = 1371/7431/0/8802
Classes: train/test/unassigned/total = 912/912/0/912
Samples: train only/test only        = 0/0
Classes: train only/test only/joint  = 0/0/912

Fraction of train set     = 15.58%
Fraction of test set only = 0.00%


In [28]:
df_training, df_val = df_of_training.loc[idx_train], df_of_training.loc[idx_test]
print(len(df_training))
print(len(df_val))

1371
7431


In [29]:
subtraining = training.get_subset(df_training.index)

In [30]:
# trainloader = torch.utils.data.DataLoader(training, batch_size=batch_size, shuffle=True)
# testloader = torch.utils.data.DataLoader(test, batch_size=batch_size)

### 模型预处理

In [31]:
print(mega)

SwinTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
  )
  (layers): Sequential(
    (0): SwinTransformerStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): SwinTransformerBlock(
          (norm1): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
          (attn): WindowAttention(
            (qkv): Linear(in_features=192, out_features=576, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=192, out_features=192, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (softmax): Softmax(dim=-1)
          )
          (drop_path1): Identity()
          (norm2): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=192, out_features=768, bias=True)
            (act): GELU(approximate='none')
            (

In [32]:
mega.head.drop = nn.Dropout(p=0.3)
mega.head.fc = nn.Linear(1536, 1024)

In [33]:
for name, param in mega.named_parameters():
    param.requires_grad = False
    if "norm.weight" == name:
        param.requires_grad = True
    if "norm.bias" == name:
        param.requires_grad = True
    if "layers.3" in name:
        param.requires_grad = True
    if "layers.2.blocks" in name:
        param.requires_grad = True
    if "head.fc" in name:
        param.requires_grad = True
for name, param in mega.named_parameters():
    print(f"{name}: {param.requires_grad}")

patch_embed.proj.weight: False
patch_embed.proj.bias: False
patch_embed.norm.weight: False
patch_embed.norm.bias: False
layers.0.blocks.0.norm1.weight: False
layers.0.blocks.0.norm1.bias: False
layers.0.blocks.0.attn.relative_position_bias_table: False
layers.0.blocks.0.attn.qkv.weight: False
layers.0.blocks.0.attn.qkv.bias: False
layers.0.blocks.0.attn.proj.weight: False
layers.0.blocks.0.attn.proj.bias: False
layers.0.blocks.0.norm2.weight: False
layers.0.blocks.0.norm2.bias: False
layers.0.blocks.0.mlp.fc1.weight: False
layers.0.blocks.0.mlp.fc1.bias: False
layers.0.blocks.0.mlp.fc2.weight: False
layers.0.blocks.0.mlp.fc2.bias: False
layers.0.blocks.1.norm1.weight: False
layers.0.blocks.1.norm1.bias: False
layers.0.blocks.1.attn.relative_position_bias_table: False
layers.0.blocks.1.attn.qkv.weight: False
layers.0.blocks.1.attn.qkv.bias: False
layers.0.blocks.1.attn.proj.weight: False
layers.0.blocks.1.attn.proj.bias: False
layers.0.blocks.1.norm2.weight: False
layers.0.blocks.1.norm

### 测试输出

In [34]:
mega.to(device)
test_input = torch.ones(1, 3, 384, 384)
test_input = test_input.to(device)
with torch.no_grad():
    test_output = mega(test_input)
print(test_output.shape)

torch.Size([1, 1024])


### 超参数设置

In [35]:
base_lr = 8e-5

param_groups = [
    {'params': [], 'lr': base_lr * 0.0625},
    {'params': [], 'lr': base_lr * 0.125},
    {'params': [], 'lr': base_lr * 0.5},
    {'params': [], 'lr': base_lr},
]

for name, param in mega.named_parameters():
    if not param.requires_grad:
        continue
    if 'layers.2.blocks' in name:
        param_groups[0]['params'].append(param)
    elif 'layers.3' in name:
        param_groups[1]['params'].append(param)
    elif name.startswith('norm'):
        param_groups[2]['params'].append(param)
    elif name.startswith('head'):
        param_groups[3]['params'].append(param)
    else:
        pass


optimizer = optim.AdamW(param_groups, weight_decay=0.01)


for i, group in enumerate(optimizer.param_groups):
    print(f'Group {i}: lr={group["lr"]}, number of parameters={len(group["params"])}')


Group 0: lr=5e-06, number of parameters=234
Group 1: lr=1e-05, number of parameters=29
Group 2: lr=4e-05, number of parameters=2
Group 3: lr=8e-05, number of parameters=2


In [36]:
distance = distances.CosineSimilarity()
reducer = reducers.ThresholdReducer(low=0)
loss_func = losses.TripletMarginLoss(margin=Vmargin, distance=distance, reducer=reducer)
mining_func = miners.TripletMarginMiner(
    margin=Vmargin, distance=distance, type_of_triplets="semihard"
)
accuracy_calculator = AccuracyCalculator(include=("precision_at_1",), k=1)
sampler = samplers.MPerClassSampler(training_labels, m=4, batch_size=batch_size, length_before_new_iter=len(training))

In [37]:
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2, verbose=True,min_lr=1e-6)

In [38]:
import pytorch_metric_learning.utils.logging_presets as LP
from pytorch_metric_learning.trainers import TrainWithClassifier
from pytorch_metric_learning.testers import GlobalEmbeddingSpaceTester
import torch
import os

# 创建日志和模型保存目录
log_folder = "/kaggle/working/metric_learning_logs"
tensorboard_folder = "/kaggle/working/metric_learning_tensorboard"
model_folder = "/kaggle/working/metric_learning_models"

os.makedirs(log_folder, exist_ok=True)
os.makedirs(tensorboard_folder, exist_ok=True)
os.makedirs(model_folder, exist_ok=True)


record_keeper, _, _ = LP.get_record_keeper(log_folder, tensorboard_folder)
hooks = LP.get_hook_container(record_keeper)
hooks.primary_metric = "precision_at_1"
hooks.log_freq = 1


tester = GlobalEmbeddingSpaceTester(
    dataloader_num_workers=4,
    normalize_embeddings=True,
    use_trunk_output=True,
    batch_size=batch_size,
    data_device=device,
    accuracy_calculator=accuracy_calculator,
    end_of_testing_hook=hooks.end_of_testing_hook
)

# 准备数据集字典
dataset_dict = {"val": test, "fromtrain": subtraining}

# 创建epoch结束时的hook
end_of_epoch_hook = hooks.end_of_epoch_hook(
    tester, 
    dataset_dict, 
    model_folder,
    test_interval=1,
    patience=4
)


models = {"trunk": mega}
loss_funcs = {"metric_loss": loss_func}
mining_funcs = {"tuple_miner": mining_func}
optimizers = {"trunk_optimizer": optimizer}
# lr_schedulers = {"trunk_scheduler_by_iteration": scheduler}

# 创建trainer
trainer = TrainWithClassifier(
    models=models,
    optimizers=optimizers,
    batch_size=batch_size,
    loss_funcs=loss_funcs,
    mining_funcs=mining_funcs,
    dataset=training,
    sampler=sampler,
    # lr_schedulers=lr_schedulers,
    end_of_iteration_hook=hooks.end_of_iteration_hook,
    end_of_epoch_hook=end_of_epoch_hook,
    dataloader_num_workers=4,
    data_device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
)

In [39]:
trainer.train(num_epochs=20)

100%|██████████| 91/91 [01:03<00:00,  1.44it/s]


In [40]:
loss_histories = hooks.get_loss_history()
acc_histories = hooks.get_accuracy_history(tester, "val", return_all_metrics=True)

print(loss_histories)
print(acc_histories)

{'~iteration~': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218

In [41]:
torch.save(mega.state_dict(), '/kaggle/working/model_nn.pth')